# 12. Formats and reproducibility

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/12-interoperability.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=12-interoperability.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/12-interoperability.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `a5bd111d11bbd55d4b74fabb094c9c91f9582494cdcfb651a5fcda6bba6e5ded`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 12: Formats and reproducibility; abstract contour illustration](assets/12-interoperability.svg)

# 12. Formats and reproducibility

Round trips are experiments

**Learning goals:** Compare GeoJSON, FlatGeobuf, and GeoPackage; verify attributes; build a small provenance record.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=12-interoperability.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Choose a format for a reason

GeoJSON is easy to inspect and web-friendly. GeoPackage stores spatial tables in SQLite. FlatGeobuf supports indexed binary vector access. File size alone does not rank them: compatibility, schema, indexing, streaming, and editing matter too. We encode this small dataset in memory with TerrIO, save its bytes, then reopen those bytes. This portable workflow is not a streaming benchmark and must fit in available memory.

In [ ]:
source = str(DATA / "countries.geojson")
source_layer = terrio.open_vector(source).layers[0]
outputs = {}
for extension in ["geojson", "fgb", "gpkg"]:
    path = OUTPUT / f"countries.{extension}"
    path.write_bytes(source_layer.to_bytes(path.name))
    outputs[extension] = path
    print(extension, path.stat().st_size, "bytes")

## Check meaning after conversion

A feature count can agree while attributes or coordinates change. We compare names as an unordered multiset because a spatially indexed format may reorder features. This check does not cover every field, type, or geometry detail.

In [ ]:
from collections import Counter
expected = Counter(f["properties"]["ADMIN"] for f in countries()["features"])
for extension, path in outputs.items():
    layer = terrio.open_bytes(path.read_bytes(), path.name).layers[0]
    actual = Counter(f["properties"]["ADMIN"] for f in layer.to_geojson()["features"])
    assert actual == expected, extension
    print(extension, "passed count and ADMIN multiset checks")

## Measure precisely what you claim

These sizes belong to a small teaching dataset and the specific writers used here. This is not a performance benchmark. Timing a tiny file in one browser would be dominated by startup, caches, and machine differences.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(list(outputs), [p.stat().st_size/1024 for p in outputs.values()], color="#008b8b")
ax.set(ylabel="File size (KiB)", title="Natural Earth 1:110m conversion outputs")
plt.show()

## Record the experiment

A checksum identifies exact bytes, not scientific validity. Store software versions, input source/version, parameters, and interpretation limits. Do not publish signed cloud URLs or access credentials in a provenance record.

In [ ]:
import hashlib
record = {"input": json.loads((DATA / "sources.json").read_text())[0],
          "software": {"terrio": terrio.__version__, "numpy": np.__version__},
          "operation": "in-memory encode/decode; preserve original geographic coordinates and attributes",
          "outputs": {k: {"bytes": p.stat().st_size, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()} for k, p in outputs.items()},
          "checks": ["feature count", "ADMIN multiset"], "limitations": "No exhaustive geometry/schema equivalence test"}
(OUTPUT / "conversion-provenance.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record["software"], indent=2))

## Your investigation

Add a numeric attribute check and a geometry-bounds check to the round trip. Explain why identical SHA-256 values are not expected across different formats.

### Discussion guide

Compare values by a stable feature identifier with a declared tolerance. Different encodings represent equivalent information using different bytes.

## Sources and credit

- [Source 1](https://terrio.dev/formats/)
- [Source 2](https://terrio.dev/conformance/)
- [Source 3](https://www.naturalearthdata.com/about/terms-of-use/)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.